# Project Motivation

**Task** - the task is a to approximate a smooth non-linear function on a bounded 2D domain

Essentially this is a supervised regression task

As this project is for learning purposes rather pure practicality, we will be using a neural network even though there are more efficient methods of function interpolation.

### Why, can we use a neural network?
-- NEED TO ADD THAT WE CAN DO THIS USING UAT


Before we look at model architecture, training and so on, let's take a look at the data.

## Data Generation

For this project we will create synthetic data.
Here is the methodology :

We have some known smooth and non-linear function $f : [a,b]^2 \to \mathbb{R} , f(x,y)$   
where $a,b\in\mathbb{R}$

To generate the dataset, we take $n$ points $(x,y)\in[a,b]^2$
Then, we evaluate the function at these points and add some noise  
This gives us : $z_{i} = f(x_{i},y_{i}) + \epsilon_{i}\;\;1\leq i\leq n$ where $\epsilon_{i}\sim N(0,\sigma^2)\;\;i.d.d$  
This gives us the final dataset $\mathbf{D}=\{(x_{i} , y_{i} , z_{i}) \}_{i=1}^n$

This is implemented in the code below

So now we have our dataset $\mathbf{D}$, we can begin to think about the model and its implementation.

We need to : 

1) Choose a model
2) Choose a loss function
3) Choose weights that makes F a close approximation of the target function

#### (1) Choosing a model
We have already decided to use a neural network but we need to think about its structure.
For now, we will consider a simple Feed-Forward Network.

##### Feed-Forward Neural Network
<img src="fnn_diagram.png" alt="Feed-Forward Neural Network Diagram"/>

I have drawn this graph to show the general structure of an FNN.

An FNN simply consists of a set of "neurons" or "nodes" (taking inspiration from neurons in the brain).

These are modelled to be basic units of computation.
Typically, each neuron performs a weighted sum on its inputs and then adds some bias. To allow a network to capture more complex patterns, we then introduce non-linearity by applying a post activation function, giving a final output.
  
We then group a some of these neurons together, forming what we call a "layer".  
Thinking about a network in terms of its layers makes it easier when it comes to implementation.

We then take some these layers to form the network as shown above.  
The input layer just refers to the information we input into the network and does not actually perform any computation.  
The intermediate layers are called "hidden" because a user cannot directly interact with them. They perform most/all of the computation.  
Finally, we have the output layer which again doesn't perform any computation and is just the output of the network.  

How is everything connected?
Generally in an FNN, each neuron passes its output to every node in the next layer. This means that the network is "fully" connected.  
  
In different models, layers might not be sequentially arranged, the network might not be fully connected, and other modifications may be added. Additionally, we can make adjustments to model's architecture to improve the model's performance. 

How do we implement this/represent this mathematically?
  
*for convenience, i will explain everything in real numbers, but we can use different objects and so forth.*  
Representing a neuron:   
Suppose, we have $n$ inputs with the $i^{th}$ input, $x_i\in\mathbb{R}$ having weight $w_i\in\mathbb{R}$.  
We can then represent the weighted sum as $$\sum_{i=1}^{n}w_i x_i$$  
We can then add a bias term $b\in\mathbb{R}$, and then apply some activation function $\sigma:\mathbb{R}\to\mathbb{R}$
This gives us the output of a neuron $y\in\mathbb{R}$ as $$y=\sigma\big(\sum_{i=1}^n w_i x_i + b\big)$$


This is for one neuron, but what if we want to represent many neurons at once?  It would be quite messy to store and manage each neuron individually so we use linear algebra to help represent everything neatly.  
Notice that we can write the weighted sum for the $j^{th}$ node (with the $i^{th}$ input having a weight $w_ji$ for the $j^{th} node$) as $$\sum_{i=1}^n w_jix_i$$

This is actually the $i,j$ entry of some matrix multiplication $A\underline{x}$ where $A\in\mathbb{R}^{m \times n}$ and $\underline{x}\in\mathbb{R}^n$
# COMPLETE EXPLANATION

In [1]:
#Libraries needed

import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import pandas as pd

In [7]:
rand = np.random.default_rng(42)


a = -10
b = 10
n = 100

#Our target function f
def f(x,y):
    return x**2+y**2

def g(x,y):
    return np.sqrt(abs(1-x**2-y**2))

def h(x,y):
    return 1/(1+25*x**2)

def i(x,y):
    return np.sin(1/x)


xcoords = rand.uniform(low=a,high=b,size=(n,1))
ycoords = rand.uniform(low=a,high=b,size=(n,1))

zcoords = f(xcoords,ycoords)

dataset = np.concatenate((xcoords,ycoords,zcoords),axis=1)

#formatting coords for input
TestTrainSplit = 0.8
batches = 5
SplitIndex = int(TestTrainSplit*n)
TrainData  = dataset[:SplitIndex]
TestData = dataset[SplitIndex:]

noise = rand.normal(loc=0,scale=1,size=(SplitIndex,1))
TrainData[:,-1] = (TrainData[:,-1].reshape(SplitIndex,1) + noise).reshape(SplitIndex)


#scaling the dataset
mean = np.mean(TrainData,0)
SD = np.std(TrainData,0)

ScaledTrainData = (TrainData - mean)/SD

TrainDataBatches = np.split(TrainData,batches)

#Simple scatter plot for visualisation
fig = px.scatter_3d(TrainData,x=0,y=1,z=2)
fig.show()

In [ ]:
def ReLu(x):
    return np.maximum(0,x)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def swish(x):
    return x * sigmoid(x)
def MSE(labels,predictions):
    loss = np.sum((predictions - labels)**2)
    loss = loss/predictions.shape[0]
    return loss

def RMSE(labels,predictions):
    loss = MSE(labels,predictions)
    return np.sqrt(loss)


def MinMax(x,min,max):
    return (x-min)/(max-min)

def InvMinMax(x,min,max):
    return  x * (max-min) + min

def normalise(x,mean,var,epsilon=1e-5):
    return (x-mean)/np.sqrt(var+epsilon)

def InvNormalise(x,mean,var,epsilon=1e-5):
    return x * np.sqrt(var + epsilon) + mean

In [ ]:
#LayerWidths -> an arr storing the number of neurons in the (k+1)th layer (left to right)
#PostActFuncs -> an arr storing the post-act func of each layer
class network:
    def __init__(self, LayerWidths,PostActivationFuncs):
        self.LayerNum = len(LayerWidths)-1
        self.LayerWidths = LayerWidths
        self.PostActivationFuncs = PostActivationFuncs
        self.layers = []
        self.LayerOutputs = []
        for i in range(1,self.LayerNum+1):
            layeri= layer(LayerWidths[i-1],LayerWidths[i],PostActivationFuncs[i])
            self.layers.append(layeri)


    def ForwardPass(self,input):
        self.LayerOutputs.clear()
        for i in range(self.LayerNum):
           input , LayerComputations = self.layers[i].compute(input)
           self.LayerOutputs.append(LayerComputations)

        return input

class layer:
    def __init__(self,InputWidth,width,PostActFunc):
        sigma = np.sqrt(2/(InputWidth+width))
        self.width = width
        self.PostActFunc = PostActFunc
        self.InputWidth = InputWidth
        self.weights = np.array(rand.normal(0,sigma, size=(width,InputWidth)))
        self.bias = np.array(rand.normal(0,sigma, size=(width,1)))
        #stochatistcally initialise a matrix


    def compute(self,input):
        computations = {}
        LinearComputation = self.weights@input + self.bias
        computations["Linear"] = LinearComputation


        PostActComputation = self.PostActFunc(LinearComputation)
        computations["LayerOutput"] = PostActComputation
        return PostActComputation , computations

In [ ]:
#formed the netowrk
fnn = network([2,32,32,32,1],[lambda x : x,np.tanh,np.tanh,np.tanh,lambda x : x])

In [ ]:
def GradDesc(CurrVal,Gradient,LearningRate=0.001):
    return CurrVal - LearningRate*Gradient

In [ ]:
BatchLossVals = [[] for i in range(batches)]
TotalLossVals = []
epochs = int(2000)
LearningRate = 1e-1
RegParam = 0.0
loss = 0
InputShape = (2,1)


for epoch in range(epochs):
    indexes = np.arange(0,len(TrainDataBatches)-1)
    np.random.shuffle(indexes)

    for index in indexes:
        CurrentBatch = TrainDataBatches[index]
        N = CurrentBatch.shape[0]

        W4 = fnn.layers[3].weights
        b4 = fnn.layers[3].bias

        W3 = fnn.layers[2].weights
        b3 = fnn.layers[2].bias

        W2 = fnn.layers[1].weights
        b2 = fnn.layers[1].bias

        W1 = fnn.layers[0].weights
        b1 = fnn.layers[0].bias

        dW4 = np.zeros((W4.shape))
        db4 = np.zeros((b4.shape))

        dW3 = np.zeros((W3.shape))
        db3 = np.zeros((b3.shape))

        dW2 = np.zeros((W2.shape))
        db2 = np.zeros((b2.shape))

        dW1 = np.zeros((W1.shape))
        db1 = np.zeros((b1.shape))

        predictions = np.zeros((N,1))
        labels = np.zeros((N,1))

        for i in range(N):
            x = CurrentBatch[i,:-1].reshape(InputShape)
            label = CurrentBatch[i,-1]
            prediction = fnn.ForwardPass(x)

            predictions[i] = prediction
            labels[i] = label

            #layer 4
            a3_i = fnn.LayerOutputs[2]["LayerOutput"]

            dLdy_i = 2/N * (prediction - label)
            db4 += dLdy_i
            dW4 += dLdy_i * a3_i.T

            #layer 3
            a2_i = fnn.LayerOutputs[1]["LayerOutput"]

            delta3 = (W4.T * dLdy_i) * (1 - a3_i ** 2)

            dW3 += delta3 @ a2_i.T
            db3 += delta3

            #Layer 2
            a1_i = fnn.LayerOutputs[0]["LayerOutput"]

            delta2 = (W3.T @ delta3) * (1-a2_i**2)

            dW2 += delta2 @ a1_i.T
            db2 += delta2

            #Layer 1
            delta1 = (W2.T @ delta2) * (1-a1_i**2)

            dW1 += delta1 @ x.T
            db1 += delta1



        fnn.layers[3].bias = GradDesc(b4,1/N * db4,LearningRate)
        fnn.layers[3].weights = GradDesc(W4,1/N * dW4,LearningRate)

        fnn.layers[2].bias = GradDesc(b3,1/N * db3,LearningRate)
        fnn.layers[2].weights = GradDesc(W3,1/N * dW3,LearningRate)

        fnn.layers[1].bias = GradDesc(b2,1/N * db2,LearningRate)
        fnn.layers[1].weights = GradDesc(W2,1/N * dW2,LearningRate)

        fnn.layers[0].bias = GradDesc(b1,1/N * db1,LearningRate)
        fnn.layers[0].weights = GradDesc(W1,1/N * dW1,LearningRate)


        loss =  MSE(labels,predictions)
        BatchLossVals[index].append(loss)
        TotalLossVals.append(loss)

    print(f"epoch: {epoch}, loss: {loss}")













In [ ]:
for layer in fnn.layers:
    print(f"weight : {layer.weights}")

In [ ]:
for j in range(len(BatchLossVals)):
    plt.plot([i for i in range(len(BatchLossVals[j]))],BatchLossVals[j])
plt.show()

In [ ]:
plt.plot(np.arange(0,len(TotalLossVals)),TotalLossVals)
plt.show()

In [ ]:
TestPredictions = np.zeros((TestData.shape[0],1))

for i in range(TestData.shape[0]):
    x = TestData[i,:-1].reshape(InputShape)
    TestPrediction = fnn.ForwardPass(x)
    TestPredictions[i] = TestPrediction
    print(f"prediction : {TestPrediction[0][0]}, actual value : {TestData[i,-1]}")


TestLabels = TestData[:,-1].reshape(TestData.shape[0],1)
TestLoss = RMSE(TestLabels,TestPredictions)

print(f"RMSE on test is {TestLoss}")

In [24]:
## Linear regression

import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.metrics import mean_squared_error

X_train = TrainData[:,:-1]
y_train = TrainData[:,-1]
X_test = TestData[:,:-1]
y_test = TestData[:,-1]
# baselines
poly = LinearRegression().fit(PolynomialFeatures(2).fit_transform(X_train), y_train)
gp = GaussianProcessRegressor(kernel=RBF(length_scale=0.5)).fit(X_train, y_train)

print("Poly MSE:", mean_squared_error(y_test, poly.predict(PolynomialFeatures(2).fit_transform(X_test))))
print("GP MSE:  ", mean_squared_error(y_test, gp.predict(X_test)))
# then compare your NN's MSE to these

Poly MSE: 0.07730026945557783
GP MSE:   111.04381375810642


In [20]:
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(42)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(2,4) 
        self.l2 = nn.Linear(4,4)
        self.l3 = nn.Linear(4,4)
        self.l4 = nn.Linear(4,1)

    def forward(self,x):
        x = x.to(dtype=torch.float32)
        x = F.tanh(self.l1(x))
        x = F.tanh(self.l2(x))
        x = F.tanh(self.l3(x))
        x = self.l4(x)
        return x





model = NeuralNetwork()

In [21]:
#turning the dataset into tensors

X_train = torch.tensor(X_train,dtype=torch.float32)
y_train = torch.tensor(y_train,dtype=torch.float32)
X_test = torch.tensor(X_test,dtype=torch.float32)
y_test = torch.tensor(y_test,dtype=torch.float32)


loss_func = nn.MSELoss()
optmizer = torch.optim.SGD(model.parameters(),lr=0.001)


C:\Users\arhaa\AppData\Local\Temp\ipykernel_34704\1406981962.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train,dtype=torch.float32)
C:\Users\arhaa\AppData\Local\Temp\ipykernel_34704\1406981962.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train,dtype=torch.float32)
C:\Users\arhaa\AppData\Local\Temp\ipykernel_34704\1406981962.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test,dtype=torch.float32)
C:\Users\arhaa\AppData\Local\Temp\ipykernel_34704\1

In [23]:
epochs = 1000
torch_net_loss = []

for i in range(epochs):
    y_pred = model.forward(X_train)

    loss = loss_func(y_pred,y_train)
    torch_net_loss.append(loss.detach().numpy())

    if i % 10 == 0: print(f'Epoch {i} and loss : {loss}')

    optmizer.zero_grad()
    loss.backward()
    optmizer.step()

Epoch 0 and loss : 1896.111572265625
Epoch 10 and loss : 1896.111572265625
Epoch 20 and loss : 1896.111572265625
Epoch 30 and loss : 1896.111572265625
Epoch 40 and loss : 1896.1114501953125
Epoch 50 and loss : 1896.111572265625
Epoch 60 and loss : 1896.111572265625
Epoch 70 and loss : 1896.111572265625
Epoch 80 and loss : 1896.1114501953125
Epoch 90 and loss : 1896.1114501953125
Epoch 100 and loss : 1896.111572265625
Epoch 110 and loss : 1896.111572265625
Epoch 120 and loss : 1896.111572265625
Epoch 130 and loss : 1896.1114501953125
Epoch 140 and loss : 1896.111572265625
Epoch 150 and loss : 1896.1114501953125
Epoch 160 and loss : 1896.1114501953125
Epoch 170 and loss : 1896.111572265625
Epoch 180 and loss : 1896.1114501953125
Epoch 190 and loss : 1896.111572265625
Epoch 200 and loss : 1896.111572265625
Epoch 210 and loss : 1896.111572265625
Epoch 220 and loss : 1896.111572265625
Epoch 230 and loss : 1896.111572265625
Epoch 240 and loss : 1896.111572265625
Epoch 250 and loss : 1896.111